In [2]:
import os
import sys
import time
import datetime
import pickle
import numpy as np
import pandas as pd 
# Add src to path
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pymc as pm
import pytensor
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

from pytensor.graph.op import Op
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap

In [12]:
DATA_PATH = "example4_new/addm_data_20251015-163921.pkl" 
data = pickle.load(open(DATA_PATH, "rb"))
DATA_TYPE = np.float64
a = data["a"]
b = data["b"]
x0 = data["x0"]
# mu1_true = data["mu1"]
# mu2_true = data["mu2"]
eta_true = data["eta"]
kappa_true = data["kappa"]
r1_data = data["r1_data"]
r2_data = data["r2_data"]
flag_full = data["flag_data"].astype(np.int32)

sigma = data["sigma"]
T = data["T"]

mu1_true_data = kappa_true * (r1_data - eta_true * r2_data)
mu2_true_data = kappa_true * (eta_true * r1_data - r2_data)

mu_true_data = data["mu_array_padded_data"].astype(DATA_TYPE)
sacc_data = data["sacc_array_padded_data"].astype(DATA_TYPE)
length_full = data["d_data"].astype(np.int32)
rt_data = data["decision_data"][:, 0].astype(DATA_TYPE)
choice_data = data["decision_data"][:, 1].astype(np.int32)

num_data, max_d = mu_true_data.shape

In [27]:
print(sacc_full[0])

[0.         0.56294376 1.35037886 2.07224015 2.30662596 3.1307183
 3.32310599 0.         0.         0.         0.         0.        ]


In [6]:
monk_data_1 = pd.read_csv("monkey-data/behav_table_1.csv")
monk_data_1.sort_values(by='trialNum', ascending=True, inplace=True)
monk_data_1


,trialNum,lVal,rVal,firstVal,secondVal,chosenVal,leftChosen,firstChosen,liftRT,reachRT,firstIsLeft,firstIsRight,secondIsLeft,secondIsRight,firstFixLat,secondFixLat,thirdFixLat,firstFixDur,secondFixDur,thirdFixDur
308,1,3,1,3,1,1,0,0,0.757170,0.172254,1,0,0,1,0.162,0.410,NaN,0.188,0.644,NaN
412,2,4,3,4,3,4,1,1,0.747137,0.150739,1,0,0,1,0.168,0.420,0.636,0.192,0.156,0.576
153,3,2,4,2,4,4,0,0,0.747201,0.151524,1,0,0,1,0.134,0.402,NaN,0.212,0.916,NaN
361,4,3,3,3,3,3,1,1,0.947236,0.141216,1,0,0,1,0.158,0.386,0.758,0.176,0.328,0.688
525,5,5,5,5,5,5,0,1,1.076900,0.164868,0,1,1,0,0.142,0.370,0.882,0.188,0.456,0.656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,583,4,2,2,4,4,1,0,0.777188,0.174139,0,1,1,0,0.171,0.423,NaN,0.208,0.656,NaN
306,584,4,3,3,4,4,1,0,0.877214,0.154446,0,1,1,0,0.173,0.401,NaN,0.184,0.780,NaN
411,585,2,3,3,2,3,0,1,1.027219,0.175890,0,1,1,0,0.117,0.405,0.765,0.228,0.316,0.552
307,586,3,4,3,4,4,0,0,0.867116,0.176373,1,0,0,1,0.149,0.417,NaN,0.220,1.048,NaN


## Transform behavioral data 

In [ ]:
DATA_TYPE = np.float64
# max_d = 12 

# Hypothetical param values 
a = 2.0 
b = 0.3
x0 = -0.2
kappa = 0.5 
eta = 0.7

r1 = monk_data_1['lVal'].to_numpy(dtype=DATA_TYPE) # r1 is left value 
r2 = monk_data_1['rVal'].to_numpy(dtype=DATA_TYPE) # r2 is right value 

# Determine length data 
third_present = monk_data_1['thirdFixLat'].notna().to_numpy()
length_data = np.where(third_present, 3, 2).astype(np.int32)

t0 = monk_data_1["firstFixLat"].to_numpy(dtype=DATA_TYPE)
t1 = monk_data_1["secondFixLat"].to_numpy(dtype=DATA_TYPE)
t2 = monk_data_1["thirdFixLat"].fillna(0).to_numpy(dtype=DATA_TYPE)

# Build padded saccade array 
sacc = np.zeros((len(monk_data_1), max_d), dtype=DATA_TYPE)
sacc[:,0] = 0.0 
sacc[:,1] = t1-t0
sacc[third_present, 2] = t2[third_present]-t0[third_present]

# Build flag data -- 0 if first fixation is left, 1 if first fixation is right
first_is_left = monk_data_1['firstIsLeft'].to_numpy().astype(int)
flag_data = np.where(first_is_left == 1,0,1).astype(np.int32)

rt_raw = monk_data_1["liftRT"].to_numpy(dtype=DATA_TYPE)
rt = rt_raw - t0 # shift RT to be relative to first fixation 
left_chosen = monk_data_1["leftChosen"].to_numpy().astype(int)
choice = np.where(left_chosen == 1, 1.0, -1.0).astype(np.int32)

mu1 = kappa * (r1 - eta * r2) 
mu2 = kappa * (eta * r1 - r2)

In [29]:
sacc

array([[0.   , 0.248, 0.   , ..., 0.   , 0.   , 0.   ],
       [0.   , 0.252, 0.468, ..., 0.   , 0.   , 0.   ],
       [0.   , 0.268, 0.   , ..., 0.   , 0.   , 0.   ],
       ...,
       [0.   , 0.288, 0.648, ..., 0.   , 0.   , 0.   ],
       [0.   , 0.268, 0.   , ..., 0.   , 0.   , 0.   ],
       [0.   , 0.26 , 0.6  , ..., 0.   , 0.   , 0.   ]], shape=(587, 12))

In [31]:
# Evaluate likelihood 
from efficient_fpt.multi_stage_cy import compute_loss_parallel, print_num_threads

print_num_threads()
num_iter = 5 
start_time = time.time()
for _ in range(num_iter):
    loss = compute_loss_parallel(
        mu1, 
        mu2, 
        rt, 
        choice,
        flag_data,
        sacc,
        length_data, 
        max_d, 
        data['sigma'], 
        a, 
        b,
        x0,
        num_threads=48,
    )
    print(loss)
end_time = time.time()
print(f"avg time per iter:", (end_time - start_time) / num_iter)

Number of available threads: 48
0.9931647896991385
0.9931647896991385
0.9931647896991387
0.9931647896991387
0.9931647896991388
avg time per iter: 0.11259322166442871


## Create Pytensor Op